# Análise Exploratória de Dados — Preços de Carros no Brasil

Notebook preparado para o exercício com a base `precos_carros_brasil.csv`.

## Tarefas
a. Carregar a base
b. Verificar valores faltantes e aplicar tratativa
c. Verificar dados duplicados
d. Separar colunas numéricas e categóricas e imprimir estatísticas descritivas
e. Imprimir contagem de valores por `model` e `brand`
f. Apresentar uma breve explicação final sobre os principais resultados


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 120)


In [3]:
# a) Carregar a base
arquivo = 'precos_carros_brasil.csv'
df = pd.read_csv(arquivo)

print('Base carregada com sucesso.')
print(f'Linhas: {df.shape[0]} | Colunas: {df.shape[1]}')
display(df.head())


Base carregada com sucesso.
Linhas: 267542 | Colunas: 11


C:\Users\efelix\AppData\Local\Temp\ipykernel_5800\4286628314.py:3: DtypeWarning: Columns (1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo)


,year_of_reference,month_of_reference,fipe_code,authentication,brand,model,fuel,gear,engine_size,year_model,avg_price_brl
0,2021.0,January,004001-0,cfzlctzfwrcp,GM - Chevrolet,Corsa Wind 1.0 MPFI / EFI 2p,Gasoline,manual,1,2002.0,9162.0
1,2021.0,January,004001-0,cdqwxwpw3y2p,GM - Chevrolet,Corsa Wind 1.0 MPFI / EFI 2p,Gasoline,manual,1,2001.0,8832.0
2,2021.0,January,004001-0,cb1t3xwwj1xp,GM - Chevrolet,Corsa Wind 1.0 MPFI / EFI 2p,Gasoline,manual,1,2000.0,8388.0
3,2021.0,January,004001-0,cb9gct6j65r0,GM - Chevrolet,Corsa Wind 1.0 MPFI / EFI 2p,Alcohol,manual,1,2000.0,8453.0
4,2021.0,January,004003-7,g15wg0gbz1fx,GM - Chevrolet,Corsa Pick-Up GL/ Champ 1.6 MPFI / EFI,Gasoline,manual,"1,6",2001.0,12525.0


In [4]:
# Visão geral da estrutura da base
print('Informações gerais da base:')
df.info()


Informações gerais da base:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267542 entries, 0 to 267541
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   year_of_reference   202297 non-null  float64
 1   month_of_reference  202297 non-null  object 
 2   fipe_code           202297 non-null  object 
 3   authentication      202297 non-null  object 
 4   brand               202297 non-null  object 
 5   model               202297 non-null  object 
 6   fuel                202297 non-null  object 
 7   gear                202297 non-null  object 
 8   engine_size         202297 non-null  object 
 9   year_model          202297 non-null  float64
 10  avg_price_brl       202297 non-null  float64
dtypes: float64(3), object(8)
memory usage: 22.5+ MB


In [5]:
# b) Verificar valores faltantes
faltantes = df.isnull().sum()
faltantes_percentual = (df.isnull().mean() * 100).round(2)

resumo_faltantes = pd.DataFrame({
    'qtd_faltantes': faltantes,
    'percentual_faltantes': faltantes_percentual
}).sort_values(by='qtd_faltantes', ascending=False)

print('Resumo de valores faltantes:')
display(resumo_faltantes)


Resumo de valores faltantes:


,qtd_faltantes,percentual_faltantes
year_of_reference,65245,24.39
month_of_reference,65245,24.39
fipe_code,65245,24.39
authentication,65245,24.39
brand,65245,24.39
model,65245,24.39
fuel,65245,24.39
gear,65245,24.39
engine_size,65245,24.39
year_model,65245,24.39


In [6]:
# Tratativa de valores faltantes
# Estratégia:
# - Colunas categóricas: preencher com 'Nao informado'
# - Colunas numéricas: preencher com a mediana

df_tratado = df.copy()

colunas_numericas = df_tratado.select_dtypes(include=['number']).columns.tolist()
colunas_categoricas = df_tratado.select_dtypes(include=['object', 'category']).columns.tolist()

for col in colunas_numericas:
    if df_tratado[col].isnull().sum() > 0:
        df_tratado[col] = df_tratado[col].fillna(df_tratado[col].median())

for col in colunas_categoricas:
    if df_tratado[col].isnull().sum() > 0:
        df_tratado[col] = df_tratado[col].fillna('Nao informado')

print('Valores faltantes após tratamento:')
display(df_tratado.isnull().sum())


Valores faltantes após tratamento:


year_of_reference     0
month_of_reference    0
fipe_code             0
authentication        0
brand                 0
model                 0
fuel                  0
gear                  0
engine_size           0
year_model            0
avg_price_brl         0
dtype: int64

In [7]:
# c) Verificar dados duplicados
qtd_duplicados = df_tratado.duplicated().sum()
print(f'Quantidade de linhas duplicadas: {qtd_duplicados}')

if qtd_duplicados > 0:
    df_tratado = df_tratado.drop_duplicates().reset_index(drop=True)
    print('Duplicados removidos com sucesso.')
else:
    print('Não há duplicados para remover.')

print(f'Nova dimensão da base: {df_tratado.shape}')


Quantidade de linhas duplicadas: 65246
Duplicados removidos com sucesso.
Nova dimensão da base: (202296, 11)


In [8]:
# d) Separar colunas numéricas e categóricas
colunas_numericas = df_tratado.select_dtypes(include=['number']).columns.tolist()
colunas_categoricas = df_tratado.select_dtypes(include=['object', 'category']).columns.tolist()

print('Colunas numéricas:')
print(colunas_numericas)
print('\nColunas categóricas:')
print(colunas_categoricas)


Colunas numéricas:
['year_of_reference', 'year_model', 'avg_price_brl']

Colunas categóricas:
['month_of_reference', 'fipe_code', 'authentication', 'brand', 'model', 'fuel', 'gear', 'engine_size']


In [9]:
# Estatística descritiva das variáveis numéricas
print('Resumo estatístico das variáveis numéricas:')
display(df_tratado[colunas_numericas].describe())


Resumo estatístico das variáveis numéricas:


,year_of_reference,year_model,avg_price_brl
count,202296.000000,202296.000000,202296.000000
mean,2021.564697,2011.271518,52756.692901
std,0.571903,6.376225,51628.794894
min,2021.000000,2000.000000,6647.000000
25%,2021.000000,2006.000000,22855.000000
50%,2022.000000,2012.000000,38027.000000
75%,2022.000000,2016.000000,64064.000000
max,2023.000000,2023.000000,979358.000000


In [10]:
# Estatística descritiva das variáveis categóricas
print('Resumo estatístico das variáveis categóricas:')
display(df_tratado[colunas_categoricas].describe())


Resumo estatístico das variáveis categóricas:


,month_of_reference,fipe_code,authentication,brand,model,fuel,gear,engine_size
count,202296,202296,202296,202296,202296,202296,202296,202296
unique,13,2092,202296,7,2113,4,3,30
top,January,003281-6,cfzlctzfwrcp,Fiat,Focus 1.6 S/SE/SE Plus Flex 8V/16V 5p,Gasoline,manual,"1,6"
freq,24260,425,1,44962,425,168684,161883,47420


In [11]:
# e) Contagem de valores por modelo e marca
contagem_modelo = df_tratado['model'].value_counts().reset_index()
contagem_modelo.columns = ['model', 'quantidade']

contagem_marca = df_tratado['brand'].value_counts().reset_index()
contagem_marca.columns = ['brand', 'quantidade']

print('Contagem por modelo:')
display(contagem_modelo)

print('Contagem por marca:')
display(contagem_marca)


Contagem por modelo:


,model,quantidade
0,Focus 1.6 S/SE/SE Plus Flex 8V/16V 5p,425
1,Palio Week. Adv/Adv TRYON 1.8 mpi Flex,425
2,Saveiro 1.6 Mi/ 1.6 Mi Total Flex 8V,400
3,Focus 2.0 16V/SE/SE Plus Flex 5p Aut.,400
4,Golf 2.0/ 2.0 Mi Flex Aut/Tiptronic.,375
...,...,...
2108,KICKS Active 1.6 16V Flex Aut.,2
2109,Saveiro Robust 1.6 Total Flex 16V,2
2110,Polo Track 1.0 Flex 12V 5p,2
2111,Gol Last Edition 1.0 Flex 12V 5p,2


Contagem por marca:


,brand,quantidade
0,Fiat,44962
1,VW - VolksWagen,44312
2,GM - Chevrolet,38590
3,Ford,33150
4,Renault,29191
5,Nissan,12090
6,Nao informado,1


In [12]:
# f) Breve explicação final
print('Resumo final da análise:')
print('A base foi carregada e inspecionada quanto a valores faltantes e duplicados.')
print('Os valores ausentes foram tratados com mediana nas variáveis numéricas e com "Nao informado" nas categóricas.')
print('Também foram identificadas as distribuições das variáveis numéricas e categóricas, além da frequência por modelo e marca.')
print('Essas análises ajudam a entender a composição da base e a preparar os dados para análises futuras.')


Resumo final da análise:
A base foi carregada e inspecionada quanto a valores faltantes e duplicados.
Os valores ausentes foram tratados com mediana nas variáveis numéricas e com "Nao informado" nas categóricas.
Também foram identificadas as distribuições das variáveis numéricas e categóricas, além da frequência por modelo e marca.
Essas análises ajudam a entender a composição da base e a preparar os dados para análises futuras.
